# 01 Build Modality Benchmark

            Objective: convert the reviewed seed capabilities into controlled modality minimal pairs and gold labels for the two tasks.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)

PROJECT_ROOT, CONFIG_PATH


## Load Reviewed Seeds


In [ ]:
target_count = int(CONFIG["project"]["target_seed_count"])
            review_path = PROJECT_ROOT / "data/processed/seeds_review.csv"
            seeds = eu.load_reviewed_seeds(review_path, target_count=target_count, strict=True)
            print(f"Loaded {len(seeds)} reviewed seeds.")
            seeds[:2]


## Generate Four Modality Variants Per Seed


In [ ]:
benchmark = eu.build_benchmark_items(seeds)
            benchmark_path = PROJECT_ROOT / "data/processed/benchmark_items.csv"
            eu.write_csv_rows(benchmark_path, benchmark)
            print(f"Wrote benchmark: {benchmark_path}")
            print(f"Items: {len(benchmark)}")
            benchmark[:4]


## Label and Shape Checks


In [ ]:
expected_items = target_count * len(eu.MODALITIES)
            assert len(benchmark) == expected_items, (len(benchmark), expected_items)
            assert len({row["item_id"] for row in benchmark}) == expected_items
            assert all(row["task1_gold_decision"] == ("yes" if row["source_modality"] == "mandatory" else "no") for row in benchmark)
            assert all(row["task2_gold_modality"] == row["source_modality"] for row in benchmark)
            assert eu.ORDINAL_STRENGTH["mandatory"] > eu.ORDINAL_STRENGTH["recommended"] > eu.ORDINAL_STRENGTH["optional"] > eu.ORDINAL_STRENGTH["nice_to_have"]
            print("OK: benchmark shape, labels, and modality ordering are valid.")
